In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, col, when, coalesce, split, explode, array

In [2]:
spark = SparkSession.builder.appName("ColumnOperations").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/19 19:53:42 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/19 19:53:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/19 19:53:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 19:53:55 WARN Utils: Service 'SparkUI'

In [3]:
data = [
    (1, "Иван Петров", "Москва,Санкт-Петербург", 30, 50000, None),
    (2, "Анна Сидорова", "Казань", 25, 60000, "Менеджер"),
    (3, "Елена Иванова", "Новосибирск,Екатеринбург,Омск", 35, 70000, "Инженер"),
    (4, "Петр Смирнов", None, 40, 80000, "Директор"),
    (5, "Мария Кузнецова", "Владивосток", 28, 55000, None)
]

In [4]:
schema = ["id", "full_name", "cities", "age", "salary", "position"]
df = spark.createDataFrame(data, schema)
df.show()

+---+---------------+--------------------+---+------+--------+
| id|      full_name|              cities|age|salary|position|
+---+---------------+--------------------+---+------+--------+
|  1|    Иван Петров|Москва,Санкт-Пете...| 30| 50000|    NULL|
|  2|  Анна Сидорова|              Казань| 25| 60000|Менеджер|
|  3|  Елена Иванова|Новосибирск,Екате...| 35| 70000| Инженер|
|  4|   Петр Смирнов|                NULL| 40| 80000|Директор|
|  5|Мария Кузнецова|         Владивосток| 28| 55000|    NULL|
+---+---------------+--------------------+---+------+--------+



In [28]:
df_greeting = df. \
    select("id", "full_name", "cities", "age", "salary", "position"). \
    withColumn("greeting", concat(lit("Привет, "), col("full_name"), lit("! Ваш ID: "), col("id").cast("string")))
df_greeting.show(truncate=False)

+---+---------------+-----------------------------+---+------+--------+----------------------------------+
|id |full_name      |cities                       |age|salary|position|greeting                          |
+---+---------------+-----------------------------+---+------+--------+----------------------------------+
|1  |Иван Петров    |Москва,Санкт-Петербург       |30 |50000 |NULL    |Привет, Иван Петров! Ваш ID: 1    |
|2  |Анна Сидорова  |Казань                       |25 |60000 |Менеджер|Привет, Анна Сидорова! Ваш ID: 2  |
|3  |Елена Иванова  |Новосибирск,Екатеринбург,Омск|35 |70000 |Инженер |Привет, Елена Иванова! Ваш ID: 3  |
|4  |Петр Смирнов   |NULL                         |40 |80000 |Директор|Привет, Петр Смирнов! Ваш ID: 4   |
|5  |Мария Кузнецова|Владивосток                  |28 |55000 |NULL    |Привет, Мария Кузнецова! Ваш ID: 5|
+---+---------------+-----------------------------+---+------+--------+----------------------------------+



In [12]:
df_salary_grade = (
    df
        .withColumn("salary_grade", when(col("salary") > 60000, "Высокая").otherwise(
            when(col("salary").between(50000, 60000), "Средняя").otherwise("Низкая")
        ))
)
df_salary_grade.show()

+---+---------------+--------------------+---+------+--------+------------+
| id|      full_name|              cities|age|salary|position|salary_grade|
+---+---------------+--------------------+---+------+--------+------------+
|  1|    Иван Петров|Москва,Санкт-Пете...| 30| 50000|    NULL|     Средняя|
|  2|  Анна Сидорова|              Казань| 25| 60000|Менеджер|     Средняя|
|  3|  Елена Иванова|Новосибирск,Екате...| 35| 70000| Инженер|     Высокая|
|  4|   Петр Смирнов|                NULL| 40| 80000|Директор|     Высокая|
|  5|Мария Кузнецова|         Владивосток| 28| 55000|    NULL|     Средняя|
+---+---------------+--------------------+---+------+--------+------------+



In [14]:
df_employee_info = (
    df
        .withColumn("employee_info", coalesce(col("position"), lit("Сотрудник")))
)
df_employee_info.show()

+---+---------------+--------------------+---+------+--------+-------------+
| id|      full_name|              cities|age|salary|position|employee_info|
+---+---------------+--------------------+---+------+--------+-------------+
|  1|    Иван Петров|Москва,Санкт-Пете...| 30| 50000|    NULL|    Сотрудник|
|  2|  Анна Сидорова|              Казань| 25| 60000|Менеджер|     Менеджер|
|  3|  Елена Иванова|Новосибирск,Екате...| 35| 70000| Инженер|      Инженер|
|  4|   Петр Смирнов|                NULL| 40| 80000|Директор|     Директор|
|  5|Мария Кузнецова|         Владивосток| 28| 55000|    NULL|    Сотрудник|
+---+---------------+--------------------+---+------+--------+-------------+



In [25]:
df_exploded_cities = (
    df
        .select("id", "full_name", explode(split("cities", ",")).alias("city"))
)
df_exploded_cities.show()

+---+---------------+---------------+
| id|      full_name|           city|
+---+---------------+---------------+
|  1|    Иван Петров|         Москва|
|  1|    Иван Петров|Санкт-Петербург|
|  2|  Анна Сидорова|         Казань|
|  3|  Елена Иванова|    Новосибирск|
|  3|  Елена Иванова|   Екатеринбург|
|  3|  Елена Иванова|           Омск|
|  5|Мария Кузнецова|    Владивосток|
+---+---------------+---------------+



In [27]:
df_age_group = (
    df
        .withColumn("age_group", when(col("age").between(18, 30), "18-30").otherwise(
            when(col("age").between(31, 50), "31-50").otherwise("51+")
        ))
)
df_age_group.show()

+---+---------------+--------------------+---+------+--------+---------+
| id|      full_name|              cities|age|salary|position|age_group|
+---+---------------+--------------------+---+------+--------+---------+
|  1|    Иван Петров|Москва,Санкт-Пете...| 30| 50000|    NULL|    18-30|
|  2|  Анна Сидорова|              Казань| 25| 60000|Менеджер|    18-30|
|  3|  Елена Иванова|Новосибирск,Екате...| 35| 70000| Инженер|    31-50|
|  4|   Петр Смирнов|                NULL| 40| 80000|Директор|    31-50|
|  5|Мария Кузнецова|         Владивосток| 28| 55000|    NULL|    18-30|
+---+---------------+--------------------+---+------+--------+---------+



In [ ]:
spark.stop()